In [ ]:
# ============================================================
# MONTE CARLO PERMUTATION INFERENCE
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata

N_PERMUTATIONS = 50_000
BASE_SEED = 20260806

# ------------------------------------------------------------
# 1. Load the controlled country-level dataset if necessary
# ------------------------------------------------------------

CONTROLLED_DATA_FILE = Path(
    "country_level_data_with_controls.csv"
)

if "controlled_data" not in globals():

    if CONTROLLED_DATA_FILE.exists():
        controlled_data = pd.read_csv(
            CONTROLLED_DATA_FILE,
            encoding="utf-8"
        )

        print("Loaded:", CONTROLLED_DATA_FILE)

    else:
        raise RuntimeError(
            "Run the country-level controls code first. "
            "country_level_data_with_controls.csv was not found."
        )

required_columns = {
    "Documented_Support",
    "Cooperation_Entries",
    "Cooperation_Breadth",
    "Income_Group",
    "Region",
}

if not required_columns.issubset(
    controlled_data.columns
):
    missing = sorted(
        required_columns
        - set(controlled_data.columns)
    )

    raise ValueError(
        f"Required columns are missing: {missing}"
    )

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def residualise(
    values,
    control_matrix,
    control_matrix_inverse
):
    """
    Remove the component of values explained by the
    control matrix.
    """

    coefficients = (
        control_matrix_inverse @ values
    )

    return (
        values
        - control_matrix @ coefficients
    )


def residual_correlation(x_residuals, y_residuals):
    """
    Pearson correlation between two residual vectors.
    """

    denominator = np.sqrt(
        np.dot(x_residuals, x_residuals)
        * np.dot(y_residuals, y_residuals)
    )

    if denominator == 0:
        raise ValueError(
            "Residual variance is zero; "
            "correlation cannot be calculated."
        )

    return float(
        np.dot(
            x_residuals,
            y_residuals
        )
        / denominator
    )

# ------------------------------------------------------------
# 3. Permutation partial-Spearman function
# ------------------------------------------------------------

def permutation_partial_spearman(
    data,
    outcome,
    controls=None,
    permutation_strata=None,
    n_permutations=50_000,
    seed=20260806,
):
    """
    Monte Carlo permutation test for a Spearman or
    partial-Spearman correlation.

    Baseline:
        controls=[]
        permutation_strata=[]

        Support labels are permuted across the full sample.

    Controlled model:
        Support labels are permuted only within the supplied
        income, region, or income-by-region strata.

    The Monte Carlo p-value uses the plus-one correction:

        p = (extreme permutations + 1) /
            (total permutations + 1)
    """

    controls = list(controls or [])
    permutation_strata = list(
        permutation_strata or []
    )

    needed_columns = list(
        dict.fromkeys(
            [
                "Documented_Support",
                outcome,
                *controls,
                *permutation_strata,
            ]
        )
    )

    working = (
        data[needed_columns]
        .dropna()
        .reset_index(drop=True)
        .copy()
    )

    # Convert support and cooperation values to average ranks.
    support_rank = rankdata(
        working["Documented_Support"].to_numpy(),
        method="average"
    )

    outcome_rank = rankdata(
        working[outcome].to_numpy(),
        method="average"
    )

    # Build the control matrix.
    if controls:

        control_dummies = pd.get_dummies(
            working[controls],
            drop_first=True,
            dtype=float
        )

        control_matrix = np.column_stack(
            [
                np.ones(len(working)),
                control_dummies.to_numpy(),
            ]
        )

    else:

        # Intercept-only model for the baseline test
        control_matrix = np.ones(
            (len(working), 1)
        )

    control_matrix_inverse = np.linalg.pinv(
        control_matrix
    )

    # Residualise the observed ranks.
    observed_support_residuals = residualise(
        support_rank,
        control_matrix,
        control_matrix_inverse
    )

    outcome_residuals = residualise(
        outcome_rank,
        control_matrix,
        control_matrix_inverse
    )

    observed_rho = residual_correlation(
        observed_support_residuals,
        outcome_residuals
    )

    # --------------------------------------------------------
    # Define the permissible permutation groups
    # --------------------------------------------------------

    if permutation_strata:

        if len(permutation_strata) == 1:
            groupby_argument = (
                permutation_strata[0]
            )
        else:
            groupby_argument = permutation_strata

        group_indices = [
            np.asarray(indices, dtype=int)

            for indices in (
                working
                .groupby(
                    groupby_argument,
                    observed=True,
                    sort=False
                )
                .indices
                .values()
            )
        ]

    else:

        # Baseline: shuffle across the entire sample
        group_indices = [
            np.arange(
                len(working),
                dtype=int
            )
        ]

    # --------------------------------------------------------
    # Run Monte Carlo permutations
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    extreme_count = 0

    for _ in range(n_permutations):

        permuted_support_rank = (
            support_rank.copy()
        )

        # Shuffle only within the permitted strata.
        for indices in group_indices:

            permuted_support_rank[indices] = (
                rng.permutation(
                    support_rank[indices]
                )
            )

        permuted_support_residuals = residualise(
            permuted_support_rank,
            control_matrix,
            control_matrix_inverse
        )

        permuted_rho = residual_correlation(
            permuted_support_residuals,
            outcome_residuals
        )

        # Two-sided test
        if (
            abs(permuted_rho)
            >= abs(observed_rho) - 1e-15
        ):
            extreme_count += 1

    monte_carlo_p = (
        extreme_count + 1
    ) / (
        n_permutations + 1
    )

    return {
        "N": len(working),
        "Observed_rho": observed_rho,
        "Permutations": n_permutations,
        "Extreme_Count": extreme_count,
        "Monte_Carlo_p_two_sided":
            monte_carlo_p,
        "Significant_at_05":
            bool(monte_carlo_p < 0.05),
    }

# ------------------------------------------------------------
# 4. Define the baseline and controlled models
# ------------------------------------------------------------

permutation_models = {
    "Baseline": {
        "controls": [],
        "strata": [],
        "scheme":
            "Unrestricted permutation",
    },

    "Income group": {
        "controls": [
            "Income_Group"
        ],
        "strata": [
            "Income_Group"
        ],
        "scheme":
            "Permuted within income groups",
    },

    "Geographic region": {
        "controls": [
            "Region"
        ],
        "strata": [
            "Region"
        ],
        "scheme":
            "Permuted within regions",
    },

    "Income group and geographic region": {
        "controls": [
            "Income_Group",
            "Region",
        ],
        "strata": [
            "Income_Group",
            "Region",
        ],
        "scheme":
            "Permuted within income-by-region strata",
    },
}

outcomes = [
    "Cooperation_Entries",
    "Cooperation_Breadth",
]

# ------------------------------------------------------------
# 5. Run all permutation tests
# ------------------------------------------------------------

permutation_results = []

test_number = 0

for model_name, specification in (
    permutation_models.items()
):

    for outcome in outcomes:

        test_number += 1

        # A separate deterministic seed for every test
        test_seed = (
            BASE_SEED + test_number
        )

        result = permutation_partial_spearman(
            data=controlled_data,
            outcome=outcome,
            controls=specification["controls"],
            permutation_strata=specification["strata"],
            n_permutations=N_PERMUTATIONS,
            seed=test_seed,
        )

        permutation_results.append({
            "Model": model_name,
            "Outcome": outcome,
            "Permutation_Scheme":
                specification["scheme"],
            "Seed": test_seed,
            **result,
        })

        print(
            f"Completed: {model_name} — {outcome}"
        )

permutation_results = pd.DataFrame(
    permutation_results
)

# ------------------------------------------------------------
# 6. Display and save results
# ------------------------------------------------------------

display(
    permutation_results.round({
        "Observed_rho": 4,
        "Monte_Carlo_p_two_sided": 6,
    })
)

baseline_permutation_results = (
    permutation_results.loc[
        permutation_results["Model"].eq(
            "Baseline"
        )
    ].copy()
)

controlled_permutation_results = (
    permutation_results.loc[
        ~permutation_results["Model"].eq(
            "Baseline"
        )
    ].copy()
)

permutation_results.to_csv(
    "all_permutation_results.csv",
    index=False,
    encoding="utf-8"
)

baseline_permutation_results.to_csv(
    "baseline_permutation_results.csv",
    index=False,
    encoding="utf-8"
)

controlled_permutation_results.to_csv(
    "stratified_controlled_permutation_results.csv",
    index=False,
    encoding="utf-8"
)

print("\nPermutation results saved.")